# DocUVerse quickstart

Three equivalent surfaces for the same retrieval pipeline, mirroring the README. The repo ships a 10-passage / 5-query toy corpus under [`examples/quickstart/`](../examples/quickstart/); each cell below produces the same ranked results.

**Prerequisite**: `pip install -e .[quickstart]` from the repo root. No external service needed — Milvus-Lite runs in-process at `./.docuverse/milvus.db`.

## Setup

The recipes use relative paths (`examples/quickstart/...`), so we run from the repo root.

In [1]:
import os
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
print(f"cwd: {Path.cwd()}")

cwd: /Users/raduflorian/sandbox2/docuverse


## 1. Python — preset with overrides (headline API)

`SearchEngine.from_preset(name, **overrides)` is the most ergonomic surface: pick a shipped recipe, override the keys you care about, get a configured engine. Discover names via `SearchEngine.list_presets()` or `docuverse presets list`.

In [2]:
from docuverse import SearchEngine

engine = SearchEngine.from_preset(
    "milvus-dense",
    model_name="ibm-granite/granite-embedding-small-english-r2",
    index_name="docuverse_quickstart",
    input_passages="examples/quickstart/passages.jsonl",
    input_queries="examples/quickstart/queries.jsonl",
    output_file="examples/quickstart/output.json",
)
engine.ingest(engine.read_data())
queries = engine.read_questions()
results = engine.search(queries)
print(engine.compute_score(queries, results))

We're on a Mac !!
Retrieval engine: milvus-dense
Running on the mps 


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

Floating point types used in the model ibm-granite/granite-embedding-small-english-r2: {torch.float32}
=== done initializing model
Cache filename is /Users/raduflorian/.local/share/elastic_ingestion/examples__quickstart__passages.jsonl_512_None_True_all_granite-embedding-small-english-r2.pickle.xz
Loaded 10 passages from cache (filtered 0).


/Users/raduflorian/sandbox2/docuverse/docuverse/utils/embeddings/dense_embedding_function.py:120: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = self.model.get_sentence_embedding_dimension()
/Users/raduflorian/miniforge3/envs/docu/lib/python3.12/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Are you sure you want to recreate the index docuverse_quickstart? It might take a long time!!

=== Collection docuverse_quickstart exists, dropping ===
Index params: {
  "index_type": "AUTOINDEX",
  "metric_type": "IP",
  "params": {}
}


...

Ingesting with a batch size of 40



  * Encoding data: 100%|██████████| 10/10 [00:00<00:00, 29.71it/s]

Creating data: 100%|██████████| 10/10 [00:00<00:00, 28.52it/s]it/s]

                                                                  

Reading examples/quickstart/queries.jsonl:: 5it [00:00, 14665.40it/s]A


Searching documents::   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating questions: 100%|██████████| 5/5 [00:00<00:00, 45889.54it/s]

Model                   M@1       M@3       M@5       
docuverse_quickstart    1.0       1.0       1.0       



## 2. Python — explicit YAML

When you want the config in version control instead of in code, `examples/quickstart/recipe.yaml` carries the same settings as the `from_preset` call above.

In [3]:
from docuverse import SearchEngine

engine = SearchEngine(config_or_path="examples/quickstart/recipe.yaml")
engine.ingest(engine.read_data())
queries = engine.read_questions()
print(engine.compute_score(queries, engine.search(queries)))

Retrieval engine: milvus-dense
Running on the mps 


Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

/Users/raduflorian/sandbox2/docuverse/docuverse/utils/embeddings/dense_embedding_function.py:120: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = self.model.get_sentence_embedding_dimension()


Floating point types used in the model ibm-granite/granite-embedding-small-english-r2: {torch.float32}
=== done initializing model
Cache filename is /Users/raduflorian/.local/share/elastic_ingestion/examples__quickstart__passages.jsonl_512_None_True_all_granite-embedding-small-english-r2.pickle.xz
Loaded 10 passages from cache (filtered 0).
Are you sure you want to recreate the index docuverse_quickstart? It might take a long time!!

=== Collection docuverse_quickstart exists, dropping ===
Index params: {
  "index_type": "AUTOINDEX",
  "metric_type": "IP",
  "params": {}
}


  * Encoding data:   0%|          | 0/10 [00:00<?, ?it/s]

  * Milvusing data:   0%|          | 0/10 [00:00<?, ?it/s]
                                                         

                                                    
Creating data:   0%|          | 0/10 [00:00<?, ?it/s]t/s]

Creating data: 100%|██████████| 10/10 [00:00<00:00, 128.55it/s]A

                                                         

                                                          

Ingesting with a batch size of 40


Reading examples/quickstart/queries.jsonl:: 5it [00:00, 12850.20it/s]


Searching documents::   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating questions: 100%|██████████| 5/5 [00:00<00:00, 47662.55it/s]

Model                   M@1       M@3       M@5       
docuverse_quickstart    1.0       1.0       1.0       



r## 3. CLI

Same pipeline, run as a one-liner. Inside the notebook we shell out with `!`; on a terminal you'd just type `docuverse run --config examples/quickstart/recipe.yaml`.

> Milvus-Lite is a single-writer embedded DB, so we release the Python engine first to free the lockfile on `./.docuverse/milvus.db` before the CLI subprocess opens it.

In [ ]:
# Milvus-Lite is single-writer per .db file, and the Python client from the
# previous cells still holds a handle. Point the CLI at its own .db file by
# bundling both overrides into a single `--override` (argparse's nargs="*"
# means a second `--override` would replace the first).
!docuverse run --config examples/quickstart/recipe.yaml \
    --override server=file:./.docuverse/milvus_cli.db index_name=docuverse_quickstart_cli

## Discover other presets

Eleven recipes ship with the package — Milvus (dense / sparse / hybrid / BM25 / SPLADE), Elastic (BM25 / dense / ELSER), ChromaDB, FAISS, LanceDB. Inspect them without instantiating a heavy engine:

In [ ]:
!docuverse presets list --with-engine

In [ ]:
!docuverse presets show milvus-dense

Dump a recipe to a YAML file you can copy and edit:

In [ ]:
!docuverse presets dump milvus-dense > /tmp/my-recipe.yaml && head -20 /tmp/my-recipe.yaml

Or in Python:

In [ ]:
from docuverse import SearchEngine

SearchEngine.list_presets()